# Causal encoder in four identity-filler settings (8 runs)

**Model.** `[operation slots] + [36 input cells] + [36 output queries]`.
* Causal attention over the WHOLE sequence: every position sees only earlier positions.
* All 36 answers are read at the output queries in ONE parallel pass. Nothing is generated and fed back, unlike
  Transformer AR.
* The queries carry only their output cell's (row, col) embeddings plus a learned query vector.
* Two variants: `causal_encoder` (6 untied blocks, 10.7M) and `looped_causal_encoder` (one shared block × 6, 1.8M).

| setting | slots | train k | paired with (same stream, validation and test set) |
|---|---|---|---|
| S5_k02345 | 5 | 0,2,3,4,5 | notebook 10 (AR + bidirectional encoder) |
| S5_k0234 | 5 | 0,2,3,4 | notebook 14 (AR) |
| S4_k0234 | 4 | 0,2,3,4 | notebook 16 (AR) |
| S6_k02345 | 6 | 0,2,3,4,5 | notebook 16 (AR) |

**Fixed across all runs.** Visible `<identity>` fillers, learned positions, seed 42, 5000 × 256, the same recipe as
the AR runs, and checkpoint selection on validation digit CE over the trained k. Singletons are tested at every slot.

**Initialization.** One common initialization per architecture across the four settings. The S4/S6 slot table has
one extra row, drawn from a separate generator.

**Read-only comparison.** The AR / encoder runs are only read, never modified.

**Send back `Causal_Encoder_Review.ipynb`.** The last cell prints and downloads it.


In [ ]:
# --- 2. Bootstrap: locate or clone the PRIVATE repository (credentials are never printed or stored) ---
import base64, importlib.util, os, pathlib, subprocess, sys

REPO_URL = "https://github.com/zhexuandliu/Decompositionality.git"
REPO_BRANCH = "main"
IN_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ

def _find_repo():
    candidates = [pathlib.Path("/content/Decompositionality"), pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
    for p in candidates:
        if (p / "src" / "decompositionality").is_dir():
            return p.resolve()
    return None

REPO_DIR = _find_repo()
if REPO_DIR is None:
    token = None
    if IN_COLAB:
        try:
            from google.colab import userdata
            token = userdata.get("GITHUB_TOKEN")
        except Exception:
            token = None
    if not token:
        import getpass
        token = getpass.getpass("GitHub token with read access to the private repo (input hidden): ")
    # Pass the credential through environment-scoped git config (not the URL, not the command line,
    # not persisted in .git/config).
    _hdr = "AUTHORIZATION: basic " + base64.b64encode(f"x-access-token:{token}".encode()).decode()
    _env = dict(os.environ, GIT_TERMINAL_PROMPT="0", GIT_CONFIG_COUNT="1",
                GIT_CONFIG_KEY_0="http.https://github.com/.extraheader", GIT_CONFIG_VALUE_0=_hdr)
    REPO_DIR = pathlib.Path("/content/Decompositionality" if IN_COLAB else "Decompositionality").resolve()
    _r = subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)],
                        env=_env, capture_output=True, text=True)
    del token, _hdr, _env
    if _r.returncode != 0:
        raise RuntimeError("git clone failed (check the GITHUB_TOKEN secret / repo access):\n" + _r.stderr[-2000:])

if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))
for _mod in ("numpy", "pandas", "matplotlib", "torch"):
    if importlib.util.find_spec(_mod) is None:  # Colab already ships these; never reinstall torch/CUDA here
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", _mod], check=True)
REPO_COMMIT = subprocess.run(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print("Repository:", REPO_DIR, "| commit:", REPO_COMMIT or "unknown")


In [ ]:
# --- CONFIGURATION -------------------------------------------------------------------------------------------------
MODE = os.environ.get("DECOMP_MODE", "full")
SEED = 42
RUN_SETTINGS = ["S5_k02345", "S5_k0234", "S4_k0234", "S6_k02345"]
RUN_MODELS = ["causal_encoder", "looped_causal_encoder"]
USE_DRIVE = IN_COLAB
RESULTS_ROOT = os.environ.get("DECOMP_RESULTS_ROOT") or (
    "/content/drive/MyDrive/decompositionality_results" if USE_DRIVE else str(REPO_DIR / "results"))
DEVICE = os.environ.get("DECOMP_DEVICE", "auto")
EVAL_CHUNK_GRIDS = 100
# -----------------------------------------------------------------------------------------------------------------
if USE_DRIVE and IN_COLAB and RESULTS_ROOT.startswith("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")
import pandas as pd
from IPython.display import display
pd.set_option("display.max_rows", 500); pd.set_option("display.max_columns", 60); pd.set_option("display.width", 250)
from decompositionality import causal_encoder_exp as CE
print("results ->", CE.family_root(RESULTS_ROOT, MODE))
display(CE.init_audit(SEED))


In [ ]:
# --- Train + evaluate sequentially (resumable) ---------------------------------------------------------------------
status = CE.run_queue(RESULTS_ROOT, MODE, SEED, RUN_SETTINGS, RUN_MODELS, repo_dir=str(REPO_DIR),
                      chunk_grids=EVAL_CHUNK_GRIDS, device=DEVICE)
display(status)


In [ ]:
# --- Report vs existing AR / bidirectional-encoder runs (read-only) + review export --------------------------------
rp = CE.report(RESULTS_ROOT, MODE, SEED)
print("REVIEW FILE TO SEND BACK:", rp)
if IN_COLAB:
    from google.colab import files
    files.download(str(rp))
